In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F5F771FB60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F5F78E8980>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure


_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F5F771FB60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F5F78E8980>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out

In [4]:
model.invoke("Provide details about the movie bahubali")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Bahubali. Let me start by recalling what I know. Bahubali is a big Indian movie, part of a two-part series. It\'s in Telugu, right? The director is S.S. Rajamouli, who\'s quite famous. The main actors are Prabhas, Tamannaah, Anushka Shetty, and maybe others like Rana Daggubati. \n\nThe first part was released in 2015, and the second in 2017. It\'s an epic fantasy with lots of action and political intrigue. The story is set in a kingdom called Mahishmati, and there\'s a lot of conflict between different factions. The protagonist, Bahubali, is a warrior who plays a crucial role in the kingdom\'s fate. \n\nI think there\'s a lot of mythology involved, maybe referencing Hindu epics like the Mahabharata or Ramayana. The cinematography is known for being spectacular, with grand scenes and massive sets. The music is by M.M. Keeravani, and the songs are popular. \n\nThere were some controversies too, like the budget

In [5]:
model_with_structure.invoke("Provide details about the movie bahubali")

Movie(title='Bahubali', year=2015, director='S.S. Rajamouli', rating=8.3)

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Bahubali")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Bahubali. Let me see what I need to do here. The available tool is the Movie function, which requires the title, year, director, and rating. First, I need to recall information about Bahubali. The title is definitely "Bahubali". The year it was released was 2015, I think. The director is S.S. Rajamouli, right? And the rating... maybe around 8.5 or 9 out of 10. Let me double-check these details to make sure. Yes, the first Bahubali movie came out in 2015, directed by S.S. Rajamouli. The IMDb rating is 8.3. So I should structure the function call with those parameters. All required fields are present, so the tool call should work.\n', 'tool_calls': [{'id': 'xj4kcfpr6', 'function': {'arguments': '{"director":"S.S. Rajamouli","rating":8.3,"title":"Bahubali","year":2015}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 

In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in Rupees")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Bahubali")
response

MovieDetails(title='Bahubali', year=2015, cast=[Actor(name='Prabhas', role='Bahubali'), Actor(name='Rana Daggubati', role='Bhallala Deva'), Actor(name='Tamannaah Bhatia', role='Devasena'), Actor(name='Sathyaraj', role='Amarendra Bahubali'), Actor(name='Shivaji Rao Gaikwad', role='Jagadala')], genres=['Action', 'Drama', 'Fantasy', 'Historical'], budget=2500000000.0)

In [9]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [11]:
from typing import Optional
from typing_extensions import TypedDict, Annotated

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title:  Annotated[str,   ..., "The title of the movie"]
    year:   Annotated[int,   ..., "The release year"]
    cast:   Annotated[list[Actor], ..., "List of actors and their roles"]
    genres: Annotated[list[str],   ..., "List of genres"]
    budget: Annotated[Optional[float], None, "Budget in millions USD"]  # ✅ None = optional

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Avengers")